[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chung-I/MiRA_training_course_2026/blob/main/notebooks/pyg_graph_classification.ipynb)

# Graph Classification with GNNs (PyG Tutorial)

This notebook trains a GCN with global mean pooling on the MUTAG molecular dataset.

## 1. Setup

In [ ]:
# On Colab, uncomment the next line:
# !pip install -q torch_geometric

import torch
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

print(f'PyTorch: {torch.__version__}')

PyTorch: 2.11.0+cu128

## 2. Load the MUTAG dataset

MUTAG contains 188 molecular graphs. Each node is an atom, each edge is a chemical bond. The binary label indicates mutagenicity.

In [ ]:
dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')

print(f'Graphs: {len(dataset)}')
print(f'Classes: {dataset.num_classes}')
print(f'Node features: {dataset.num_node_features}')

# Example graph
g = dataset[0]
print(f'\nExample graph: {g.num_nodes} nodes, {g.num_edges} edges, label={g.y.item()}')

Graphs: 188
Classes: 2
Node features: 7

Example graph: 17 nodes, 38 edges, label=1

## 3. GCN with global mean pooling

A 3-layer GCN computes node embeddings. `global_mean_pool` averages all node embeddings in each graph into a single vector (the readout). A linear layer predicts the graph label from that vector.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid_ch)
        self.conv2 = GCNConv(hid_ch, hid_ch)
        self.conv3 = GCNConv(hid_ch, hid_ch)
        self.lin = torch.nn.Linear(hid_ch, out_ch)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        x = global_mean_pool(x, batch)  # readout
        x = F.dropout(x, p=0.5, training=self.training)
        return self.lin(x)

accs = []
for seed in range(5):
    torch.manual_seed(seed)
    ds = dataset.shuffle()
    split = int(0.8 * len(ds))
    train_loader = DataLoader(ds[:split], batch_size=64, shuffle=True)
    test_loader = DataLoader(ds[split:], batch_size=64)

    model = GCN(dataset.num_node_features, 64, dataset.num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    model.train()
    for epoch in range(200):
        for batch in train_loader:
            optimizer.zero_grad()
            F.cross_entropy(model(batch.x, batch.edge_index, batch.batch), batch.y).backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    for batch in test_loader:
        pred = model(batch.x, batch.edge_index, batch.batch).argmax(1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    accs.append(correct / total)
    print(f'  Seed {seed}: {correct/total*100:.1f}%')

print(f'\nTest accuracy (mean of 5 seeds): {sum(accs)/len(accs)*100:.1f}%')

  Seed 0: 76.3%
  Seed 1: 71.1%
  Seed 2: 73.7%
  Seed 3: 68.4%
  Seed 4: 63.2%

Test accuracy (mean of 5 seeds): 70.5%

MUTAG uses molecular bonds as edges. The `global_mean_pool` readout averages all node embeddings in a graph into one vector, which maps to the graph-level prediction row on the task-types slide. The 70.5% accuracy is modest because MUTAG is small (188 graphs) and the 80/20 split has high variance.